In [1]:
from Strategies.Autotrader.TP_api import TP_api
import Strategies.Autotrader.enumerate as ENUM
import pandas as pd
from Database.TPData import TPData
from datetime import datetime, timezone

In [2]:
cls = TP_api('prod')
today = datetime.now()

In [15]:
ts_utc = datetime(2023, today.month, today.day, 0, 0, 0).isoformat(timespec='milliseconds')
print(ts_utc)
result_dict = cls.get_own_trades(ts_utc)

2023-10-22T00:00:00.000


In [16]:
result_dictlist = [d for d in result_dict if d['trader_name'] == '220_ETC-autotrader']

In [18]:
db_cols = {'trade_id',
           'order_id',
           'exchange',
           'execution_time',
           'state',
           'trading_portfolio',
           'price',
           'quantity',
           'buy_delivery_area',
           'sell_delivery_area',
           'product_id',
           'product_type',
           'product_name',
           'slot_type',
           'slot_information',
           'counterparty',
           'aggressor',
           'initiator',
           'aggressor_broker_id',
           'initiator_broker_id'}

In [19]:
filtered_data = [{col: data[col] for col in db_cols if col in data} for data in result_dictlist]

In [20]:
for data_dict in filtered_data:
    data_dict['algo_id'] = data_dict.pop('trading_portfolio', None)


In [21]:
df = pd.DataFrame(filtered_data)
df

,trade_id,slot_information,state,product_name,quantity,order_id,initiator,slot_type,product_id,counterparty,execution_time,exchange,aggressor_broker_id,product_type,price,sell_delivery_area,buy_delivery_area,aggressor,initiator_broker_id,algo_id
0,EDEB1122023:20231116:3:34900S,MarketMakingLimitBid,ACTI,Euro - Euro Weeks_Wk48-23_Wk48-23,1.0,51854-1700117058461032022-1700121377221980296,Y,MM_Order_BID_10641710_10000102_1144,10000102_1144,,2023-11-16T07:56:25.796000,TRAYPORT,1441,Wk48-23,99.90,10641710,,N,1441,mm_strategy_deb_dw
1,EDEB4112023:20231116:3:35200B,MarketMakingLift,ACTI,Euro - Euro Weeks_Wk47-23_Wk47-23,1.0,51851-1700117058461035574-1700121386845755816,N,LIFT_10641710,10000102_1143,,2023-11-16T07:56:26.845000,TRAYPORT,1441,Wk47-23,91.25,,10641710,Y,1441,mm_strategy_deb_dw
2,EDEB1122023:20231116:36:392000B,MarketMakingLimitAsk,ACTI,Euro - Euro Weeks_Wk48-23_Wk48-23,1.0,51854-1700117058461032032-1700132036216558912,Y,MM_Order_ASK_10641710_10000102_1144,10000102_1144,,2023-11-16T10:56:53.824000,TRAYPORT,1441,Wk48-23,99.86,,10641710,N,1441,mm_strategy_deb_dw
3,EDEB4112023:20231116:54:392500S,MarketMakingLift,ACTI,Euro - Euro Weeks_Wk47-23_Wk47-23,1.0,51851-1700117058461873490-1700132214661564544,N,LIFT_10641710,10000102_1143,,2023-11-16T10:56:54.661000,TRAYPORT,1441,Wk47-23,89.50,10641710,,Y,1441,mm_strategy_deb_dw
4,EDEB1122023:20231116:38:392700S,MarketMakingLift,ACTI,Euro - Euro Weeks_Wk48-23_Wk48-23,1.0,51854-1700117058461032217-1700132215050309750,N,LIFT_10641710,10000102_1144,,2023-11-16T10:56:55.050000,TRAYPORT,1441,Wk48-23,99.85,10641710,,Y,1441,mm_strategy_deb_dw
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
833,4905380,lift,ACTI,Euro - Euro Quarters_Q324_Q324,1.0,35443593,N,LIFT_Arb_Order_BID_1705652479,10000105_83,,2024-01-19T08:38:03.939000,TRAYPORT,7,Q324,72.75,10641710,,Y,7,arbitrage-de-qa-1
834,EDEBQ072024:20240119:335:712500B,leadBuy,ACTI,Euro - Euro Quarters_Q324_Q324,1.0,51601-1705647066518778291-1705669296953870501,Y,Arb_Order_BID_1705652479,10000105_83,,2024-01-19T13:02:39.905000,TRAYPORT,1441,Q324,73.38,,10641710,N,1441,arbitrage-de-qa-1
835,4906350,lift,ACTI,Euro - Euro Quarters_Q324_Q324,1.0,35457073,N,LIFT_Arb_Order_BID_1705652479,10000105_83,,2024-01-19T13:02:40.350000,TRAYPORT,7,Q324,73.45,10641710,,Y,7,arbitrage-de-qa-1
836,EDEBQ072024:20240119:336:713700B,leadBuy,ACTI,Euro - Euro Quarters_Q324_Q324,1.0,51601-1705647066518780209-1705669361867338472,Y,Arb_Order_BID_1705652479,10000105_83,,2024-01-19T13:02:44.634000,TRAYPORT,1441,Q324,73.38,,10641710,N,1441,arbitrage-de-qa-1


In [22]:
from Database.DB_reader import Database

db_cls = Database()
r = df.to_sql(schema='algo', name='stage_strategy_trades', if_exists='replace', con=db_cls.connection_string)
print(r)
db_cls.merge_from_staging_to_prod('algo', 'strategy_trades')

838